# Simulation-to-real readiness review

**Time:** 2 hours plus a supervised physical-cell session  
**Prerequisites:** notebooks 04–09 and a reproducible successful simulation run.

## Learning objectives

- Separate simulation evidence from physical-cell evidence
- Audit calibration, units, frames, limits, gates, latency, and recovery behavior
- Define staged, supervised acceptance tests before autonomous operation
- Produce a go/no-go record with named owners and unresolved risks

> This notebook never commands the real robot. Follow the facility's procedures and the supervising Rockwell/FANUC personnel. A green simulation report is necessary evidence, not authorization for physical motion.

In [ ]:
import json
from pathlib import Path

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / 'fanuc_er4ia_sim' / 'docs' / 'CALIBRATION_V05.md').is_file(): return p
    raise FileNotFoundError('Repository root not found')
ROOT = find_root(); SIM = ROOT / 'fanuc_er4ia_sim'; RUNS = SIM / 'runs'
runs = sorted((p for p in RUNS.glob('*') if (p / 'execution_report.json').exists()), key=lambda p: p.stat().st_mtime, reverse=True) if RUNS.exists() else []
RUN = runs[0] if runs else RUNS / 'stack_01'
print('Evidence run:', RUN)

## 1. State what simulation has not proved

The starter explicitly does **not** establish hardware transfer accuracy. Important remaining uncertainties include real camera/tool calibration after the rebuild, lens distortion omitted by the pinhole simulator, actual finger compliance and friction, controller frame conversion, real object variation, communication latency, and safety-system behavior.

The physical-controller note `sim_q3 = controller_J3 + J2` belongs only in a future, reviewed hardware interface. Simulation FK/IK already uses relative joints; applying the conversion there would be an error.

## 2. Automated simulation evidence

The next cell checks only machine-readable simulation evidence. Every physical item remains separately unverified.

In [ ]:
def load(name):
    p = RUN / name
    return json.loads(p.read_text()) if p.exists() else {}
report = load('execution_report.json'); evaluation = load('perception_evaluation.json')
cube_matches = [m for m in evaluation.get('matches', []) if m.get('category') == 'cube']
sim_checks = {
    'execution_report_present': bool(report),
    'task_complete': report.get('task_complete') is True,
    'six_verified_pickups': report.get('verified_pickups') == 6,
    'six_verified_placements': report.get('verified_placements') == 6,
    'six_cubes_matched': len(cube_matches) == 6,
    'no_missed_objects': evaluation.get('missed_objects') == [],
    'no_unmatched_detections': evaluation.get('unmatched_detections') == [],
    'manifest_present': (RUN / 'experiment_manifest.json').exists(),
}
for name, passed in sim_checks.items(): print(('PASS' if passed else 'FAIL'), name)

## 3. Human-reviewed readiness items

Assign an owner and evidence link for every item. Do not mark an item complete from memory.

### Geometry and calibration

- Confirm robot model, gripper model, 6 mm/jaw stroke, 12 mm straight-face opening, and 293 mm J6-to-tip length against the physical cell.
- Verify tool frame, user/world frame, table height, and camera extrinsics using a documented calibration target.
- Measure reprojection/position error across the usable workspace, not at one central point.
- Decide how measured lens distortion is handled on real images.

### Controls and limits

- Independently verify joint conventions, units, sign, limits, speeds, workspace boundaries, collision zones, payload, and TCP.
- Confirm E-stop, enabling device, guarding, reduced-speed mode, and the named supervisor's stop procedure.
- Confirm that stale images/plans, missing reports, low confidence, and communication timeouts cause a safe stop.

### Perception and task

- Collect representative real images for lighting, reflections, occlusion, object variation, and empty/extra-object cases.
- Evaluate without using ground truth to repair detections or generate commands.
- Measure end-to-end latency and define the maximum age of an acceptable capture/plan.

In [ ]:
physical_checks = {
    'supervisor_and_facility_approval': {'status': 'NOT_VERIFIED', 'owner': '', 'evidence': ''},
    'tool_and_base_frames_measured': {'status': 'NOT_VERIFIED', 'owner': '', 'evidence': ''},
    'camera_reprojection_test_passed': {'status': 'NOT_VERIFIED', 'owner': '', 'evidence': ''},
    'joint_units_signs_and_limits_verified': {'status': 'NOT_VERIFIED', 'owner': '', 'evidence': ''},
    'estop_guarding_and_stop_procedure_tested': {'status': 'NOT_VERIFIED', 'owner': '', 'evidence': ''},
    'reduced_speed_dry_run_passed': {'status': 'NOT_VERIFIED', 'owner': '', 'evidence': ''},
    'real_image_perception_set_passed': {'status': 'NOT_VERIFIED', 'owner': '', 'evidence': ''},
    'latency_and_stale_plan_limit_verified': {'status': 'NOT_VERIFIED', 'owner': '', 'evidence': ''},
}
readiness_record = {'simulation_run': str(RUN), 'simulation_checks': sim_checks, 'physical_checks': physical_checks}
print(json.dumps(readiness_record, indent=2))

## 4. Use staged physical validation

Advance only with the supervisor's approval and a predefined rollback/stop condition. A sensible progression is:

1. **Offline real-image replay:** capture only; run perception/planning without sending commands.
2. **Frame validation:** compare selected image points and measured fixture points across the workspace.
3. **No-object motion review:** reduced speed, generous clearance, step/single-cycle mode as directed.
4. **Single-object approach:** stop above the object and measure alignment; no grasp yet.
5. **Single pickup and safe placement:** one object, conservative speed, human confirmation between phases.
6. **Repeated single-object trials:** include varied poses and documented failures.
7. **Partial then full task:** only after preceding acceptance criteria pass.

Never jump stages because the event deadline is close.

## 5. Make the decision explicit

A **GO** requires every automated simulation check, every applicable physical check, named supervision, and no unresolved high-severity risk. Otherwise the result is **NO-GO** or **CONDITIONAL** with a precisely limited next test—not permission for autonomous operation.

In [ ]:
all_sim = all(sim_checks.values())
all_physical = all(v['status'] == 'PASS' and v['owner'] and v['evidence'] for v in physical_checks.values())
decision = 'GO' if all_sim and all_physical else 'NO-GO'
print('Decision:', decision)
print('Simulation checks complete:', all_sim)
print('Physical checks complete:', all_physical)
if decision != 'GO': print('This record does not authorize physical autonomous operation.')

## Capstone exercise

As a team, conduct a readiness review where one person presents evidence and another acts as skeptical reviewer. Produce: the reproducible simulation run, calibration evidence, real-image evaluation, staged test results, unresolved-risk list, recovery/stop rules, named owners, and supervisor sign-off.

You have completed the curriculum when the team can defend both sides of the boundary: **what the system has demonstrated, and what it has not yet demonstrated.**